# Phase 1: Inference and Panel Diagnostics

## The Hook
To accurately measure the impact of promotions across the Dominicks Finer Foods chain, we first need to understand the underlying structure of our data. Retail demand is inherently noisy and highly heterogeneous. 

## The Setup
We load the raw transactional data and product metadata to analyze cross-sectional variance (Store-to-Store differences), distribution skewness, and price elasticity. This comprehensive EDA validates our choice of **Fixed Effects**, **Log Transformations**, and specific feature engineering in the regression pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import sys

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')

# --- HIGH PRIORITY PATH CONFIGURATION ---
cwd = os.getcwd()
project_root = os.path.abspath(os.path.join(cwd, ".." if "notebooks" in cwd else "."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data.load_data import load_data
from src.data.preprocess import preprocess_data

print(f"Project root verified: {project_root}")

# Load for Panel Analysis
transactions_path = os.path.join(project_root, 'data', 'raw', 'wcer.csv')
products_path = os.path.join(project_root, 'data', 'raw', 'upccer.csv')

transactions, products = load_data(transactions_path, products_path)
df = preprocess_data(transactions, products)
df.head()

## 1. Distribution Skewness: Why We Used Log Transformations
Retail sales and prices are notoriously right-skewed. If we ran OLS on raw units, the residuals would violate homoscedasticity assumptions. Let's look at the density of Sales to justify the `log1p` approach.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sample_df = df.sample(min(100000, len(df)))

# Raw Sales Distribution
sns.histplot(sample_df['Sales'], bins=50, ax=axes[0], color='#2E86AB')
axes[0].set_title('Raw Sales (Right-Skewed)')
axes[0].set_xlabel('Units Sold')

# Log Sales Distribution
sns.histplot(np.log1p(sample_df['Sales']), bins=50, ax=axes[1], color='#E63946')
axes[1].set_title('Log(Sales) (Normalized)')
axes[1].set_xlabel('Log(Units Sold)')

plt.tight_layout()
plt.show()

## 2. Unobserved Heterogeneity: Cross-Sectional Variance
A promotion at a high-volume suburban store yields different absolute numbers than the exact same promotion at a low-volume urban store. This variance requires us to use High-Dimensional Fixed Effects to 'absorb' store baselines.

In [ ]:
top_stores = df['Store_ID'].value_counts().nlargest(15).index
df_sample_stores = df[df['Store_ID'].isin(top_stores)].copy()

plt.figure(figsize=(14, 7))
sns.boxplot(data=df_sample_stores, x='Store_ID', y='Sales', order=top_stores, palette='viridis', showfliers=False)
plt.title('Baseline Sales Variance Across Top 15 Stores')
plt.xlabel('Store ID')
plt.ylabel('Weekly Sales')
plt.xticks(rotation=45)
plt.show()

## 3. Price Elasticity and Promotional Lift
Promos don't just change a binary flag; they often represent substantial price cuts. Let's visualize the relationship between price and sales, segmented by promotional status. This validates why we include both Price and Promo (and often their interaction) in our models.

In [ ]:
plt.figure(figsize=(12, 8))
# Sample down for a readable scatter plot
scatter_df = df.sample(min(20000, len(df)), random_state=42)

sns.scatterplot(
    data=scatter_df, 
    x=np.log1p(scatter_df['Price']), 
    y=np.log1p(scatter_df['Sales']), 
    hue='Promo', 
    alpha=0.4, 
    palette={0: '#34495e', 1: '#e74c3c'}
)
plt.title('Log(Sales) vs Log(Price) Segmented by Promo')
plt.xlabel('Log(Price)')
plt.ylabel('Log(Sales)')
plt.legend(title='On Promotion', labels=['No', 'Yes'])
plt.show()

## 4. Seasonal Demand Checking
Let's look at whether there is broad monthly seasonality in demand. This tells us if we need seasonal control dummies (like Month indicator variables) in our regression, or seasonal lags in our forecasting models.

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x='Month', y=np.log1p(df['Sales']), palette='coolwarm', showfliers=False)
plt.title('Monthly Sales Variance (Log Scale)')
plt.xlabel('Month (1=Jan, 12=Dec)')
plt.ylabel('Log(Weekly Sales)')
plt.show()

print("EDA Phase 1 Complete. Variables show clear skewness, cross-sectional heterogeneity, price-sensitivity, and mild seasonality.")